# Get planet data

In [1]:
"""
Fetch heliocentric state vectors for all planets from the JPL Horizons API
and store each as a CSV: date, day, X, Y, Z, VX, VY, VZ
(positions in AU, velocities in AU/day, as returned by Horizons with OUT_UNITS='AU-D')
"""
import os
import re
import time
from datetime import datetime

import pandas as pd
import requests

# Where the output CSVs get written. Change this to whatever folder you want.
OUTPUT_DIR = '../planet_data'

# Horizons major-body IDs (heliocentric, barycenter of each planet system)
PLANETS = {
    'mercury': 199,
    'venus':   299,
    'earth':   399,
    'mars':    499,
    'jupiter': 599,
    'saturn':  699,
    'uranus':  799,
    'neptune': 899,
}

MONTHS = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
          'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}

# Same 4-line record pattern as the manual-download parser
RECORD_RE = re.compile(
    r'(?:[\d.]+) = A\.D\. (\d{4})-(\w{3})-(\d{2}) [\d:.]+ \w+\s*\n'
    r'\s*X =\s*([-\d.E+]+) Y =\s*([-\d.E+]+) Z =\s*([-\d.E+]+)\s*\n'
    r'\s*VX=\s*([-\d.E+]+) VY=\s*([-\d.E+]+) VZ=\s*([-\d.E+]+)\s*\n'
)

HORIZONS_URL = "https://ssd.jpl.nasa.gov/api/horizons.api"


def fetch_horizons_vectors(command, start, stop, step='1d', center='500@10'):
    """Query Horizons for heliocentric state vectors of `command` (body ID)."""
    params = {
        'format':      'text',
        'COMMAND':     f"'{command}'",
        'OBJ_DATA':    "'NO'",       # skip physical-data block, faster
        'MAKE_EPHEM':  "'YES'",
        'EPHEM_TYPE':  "'VECTORS'",
        'CENTER':      f"'{center}'",   # 500@10 = Sun body center
        'START_TIME':  f"'{start}'",
        'STOP_TIME':   f"'{stop}'",
        'STEP_SIZE':   f"'{step}'",
        'VEC_TABLE':   "'2'",        # state vector {x,y,z,vx,vy,vz} only
        'REF_PLANE':   "'ECLIPTIC'",
        'OUT_UNITS':   "'AU-D'",
    }
    r = requests.get(HORIZONS_URL, params=params, timeout=60)
    r.raise_for_status()
    return r.text


def parse_vectors(raw_text):
    block = raw_text.split('$$SOE')[1].split('$$EOE')[0]
    rows = []
    for m in RECORD_RE.finditer(block):
        yr, mon, day, X, Y, Z, VX, VY, VZ = m.groups()
        date = datetime(int(yr), MONTHS[mon], int(day)).strftime('%Y-%m-%d')
        rows.append((date, float(X), float(Y), float(Z), float(VX), float(VY), float(VZ)))

    df = pd.DataFrame(rows, columns=['date', 'X', 'Y', 'Z', 'VX', 'VY', 'VZ'])
    df.insert(1, 'day', range(len(df)))  # row index, 0-based
    return df


if __name__ == '__main__':
    START, STOP, STEP = '2020-01-01', '2024-01-01', '1d'

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for name, body_id in PLANETS.items():
        raw = fetch_horizons_vectors(body_id, START, STOP, STEP)
        df = parse_vectors(raw)
        out_path = os.path.join(OUTPUT_DIR, f'{name}_orbit.csv')
        df.to_csv(out_path, index=False)
        print(f"{name:8s}: {len(df):5d} rows -> {out_path}")
        time.sleep(1)  # be polite to the API between requests

mercury :  1462 rows -> ../planet_data\mercury_orbit.csv
venus   :  1462 rows -> ../planet_data\venus_orbit.csv
earth   :  1462 rows -> ../planet_data\earth_orbit.csv
mars    :  1462 rows -> ../planet_data\mars_orbit.csv
jupiter :  1462 rows -> ../planet_data\jupiter_orbit.csv
saturn  :  1462 rows -> ../planet_data\saturn_orbit.csv
uranus  :  1462 rows -> ../planet_data\uranus_orbit.csv
neptune :  1462 rows -> ../planet_data\neptune_orbit.csv
